# Module 12 Lab - Ethics, Fairness, and Bias in ML**Objective:** To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.**In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.**

## Part 1: What is Algorithmic Bias?**Concept:** Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.**Sources of Bias:***   **Historical Bias:** The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).*   **Measurement Bias:** The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).*   **Representation Bias:** The data underrepresents certain groups, so the model doesn't learn to perform well for them.**Problem:** We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like `sex` and `race`, which we can use to audit our model for bias.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Load the data
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, header=None, names=columns, sep=',\s*', engine='python', na_values='?')

# Data Cleaning
df.dropna(inplace=True)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

X = df.drop('income', axis=1)
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Create a preprocessing pipeline
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline model
model = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

Overall model accuracy: 84.61%


## Part 2: Auditing the Model for FairnessHigh overall accuracy can hide poor performance on specific subgroups. We need to audit the model by comparing its performance across sensitive attributes like `sex`.**Concept: Group Fairness**One common fairness goal is to ensure the model works equally well for different groups. We can measure this by calculating metrics for each group separately.**Your Task:** Create a function to calculate accuracy for different subgroups and then use it to compare the model's performance for males and females.

In [6]:
# --- ENTER YOUR CODE HERE ---
def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup"""
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]
    return model.score(X_subgroup, y_subgroup)

acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

print(f"Male accuracy: {acc_male:.2%}")
print(f"Female accuracy: {acc_female:.2%}")
print(f"Difference: {abs(acc_male - acc_female):.2%}")

Male accuracy: 81.20%
Female accuracy: 91.81%
Difference: 10.61%


### Task 2: Deeper Dive with a Confusion MatrixAccuracy alone doesn't tell the whole story. Let's look at the types of errors the model makes for each group.**Your Task:** Calculate and compare the **False Positive Rate (FPR)** and **False Negative Rate (FNR)** for males and females.*   **FPR:** `FP / (FP + TN)` - The percentage of people who did NOT have high income but were incorrectly predicted to have high income.*   **FNR:** `FN / (FN + TP)` - The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [5]:
from sklearn.metrics import confusion_matrix

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculate FPR and FNR using confusion matrix for subgroup"""
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]
    y_pred = model.predict(X_subgroup)
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    return fpr, fnr

fpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')

print(f"Male FPR: {fpr_male:.2%}, FNR: {fnr_male:.2%}")
print(f"Female FPR: {fpr_female:.2%}, FNR: {fnr_female:.2%}")

Male FPR: 10.26%, FNR: 37.80%
Female FPR: 2.81%, FNR: 47.84%


## 📝 Reflective Knowledge Check**Instructions:** Answer the following questions in this markdown cell. Your answers should be based on **your specific results** from the code you ran above.1.  **Analyze Your Results:** Look at the subgroup accuracies you calculated. Is there a significant difference in how the model performs for males versus females? Which group does the model perform better for?2.  **Interpret the Errors:** Compare the False Positive and False Negative rates between the two groups. For which group is the model more likely to make a False Positive error (predicting high income when it's not)? What is the real-world consequence of this specific error in the context of a loan application?3.  **Justify a Decision:** Imagine you are on an ethics board reviewing this model for use in a hiring process, where a high-income prediction is used to screen candidates for a high-paying job. Based on the specific FNR and FPR values you calculated, would you approve this model for deployment? Justify your decision by explaining which error type (FPR or FNR) is more harmful in this context and how your results show a potential disparate impact.4.  **Propose a Mitigation:** The simplest way to try and mitigate bias is to remove the sensitive feature. If you were to remove the 'sex' column from the data and retrain the model, do you think the model would become fair? Why or why not? (Hint: Think about what other columns might be correlated with 'sex').**[ENTER YOUR ANSWERS HERE]**

## Student Answers to Reflective Knowledge Check

### 1. Analyze Your Results:
- **Male accuracy: 81.20%**
- **Female accuracy: 91.81%**
- **Difference: 10.61%**

Yes, there is a significant difference in how the model performs for males versus females. The model performs **significantly better on females** (about 10.6% higher accuracy). Given the overall model accuracy of 84.61%, this 10.6% gap is substantial and indicates the model is biased in favor of female predictions.

### 2. Interpret the Errors:
- **Male FPR: 10.26%** vs **Female FPR: 2.81%**
- **Male FNR: 37.80%** vs **Female FNR: 47.84%**

The model is **more likely to make a False Positive error for males** (10.26% FPR for males vs only 2.81% for females). Males are over 3.6x more likely to be incorrectly predicted as having high income when they actually don't.

In a **loan application context**, a False Positive means a person who does NOT earn more than $50K is predicted to earn above $50K. For males, this 10.26% error rate means many males who don't actually qualify for loans would be approved, potentially leading to defaults and financial harm. Being falsely approved for a loan you can't afford is a serious consequence.

### 3. Justify a Decision:
Based on my results, I **would NOT approve this model for deployment** in a hiring process.

The key issues:
- **FNR is more harmful in a hiring context**: A False Negative means a person who truly IS high-earning is predicted as low-earning. For males, the FNR is 37.80%, meaning over 37% of males who should get the high-paying job would be denied. For females, it's even worse at 47.84%.
- This model would be **systematically denying qualified candidates** from both groups, but especially females.
- The **FPR also shows disparate impact**: Males are 3.6x more likely than females to receive a false approval, which creates an unfair advantage.
- The model fails the test of fairness across subgroups, and deploying it would likely lead to unfair hiring outcomes that perpetuate or create new inequities.

### 4. Propose a Mitigation:
Simply removing the 'sex' column would **NOT make the model fully fair**, for several reasons:

- **Other features are correlated with sex**: Columns like 'occupation', 'workclass', 'hours-per-week', and 'relationship' are all strongly correlated with gender. Certain occupations are gender-segregated, women may work fewer hours on average, etc. So even without the 'sex' feature, the model would indirectly learn about it through these proxy features.

- **Better mitigations needed**:
  1. **Feature engineering**: Remove or transform proxy features (like 'occupation') that carry indirect gender bias
  2. **Re-weighting**: Apply different class weights during training to ensure equal representation
  3. **Adversarial debiasing**: Train the model to make predictions while being unable to predict the sensitive attribute
  4. **Use fairness-aware algorithms**: Implement techniques like equalized odds or demographic parity constraints
  5. **Collect more balanced data**: Ensure training data has equal representation across all subgroups

The most effective approach would be a combination of removing the sensitive feature, removing strong proxy features, and applying fairness constraints during model training.